# PhoBERT v3: consistent profanity labels (annotation guidelines v3)

Setup: **GPU T4 x2**, **Internet On**, secret `HF_TOKEN`.
**Input:** the same private dataset as v1/v2 (the three `vfyr_*_cleaned.csv` files).

The notebook rebuilds the v2 data (leakage removal, label-error pruning, + ViHSD), then applies
`relabels/guidelines_v3.csv` (1,179 re-labeled comments) and trains PhoBERT on it.
The model is pushed to the **`v3` branch** of the Hub repo (the `main` branch, used by the demo,
is not touched), so v2 and v3 can be compared before switching.

In [ ]:
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyvi emoji cleanlab
import glob, os, shutil, json
os.makedirs("data/raw", exist_ok=True)
for split in ["train", "val", "test"]:
    matches = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) if split in os.path.basename(p).lower()]
    assert len(matches) == 1, f"Expected exactly one CSV for '{split}', found: {matches}"
    shutil.copy(matches[0], f"data/raw/{split}.csv")
os.makedirs("reports", exist_ok=True)
!python scripts/find_label_issues.py | tail -2
!python scripts/make_clean_splits.py --prune > /dev/null
!python scripts/add_vihsd.py > /dev/null
!python scripts/apply_relabels.py --src data/clean_vihsd --dst data/clean_vihsd_v3

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Logged in as:", whoami()["name"])

In [ ]:
!python scripts/train_phobert.py --data-dir data/clean_vihsd_v3 --no-class-weights --epochs 6 --output-dir models/v3 --tag _v3 --no-push

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = f"{whoami()['name']}/phobert-vi-toxic-comment"
api.create_branch(repo_id, branch="v3", exist_ok=True)
api.upload_folder(folder_path="models/v3/best", repo_id=repo_id, revision="v3", commit_message="v3: guidelines v3 labels")
print(f"https://huggingface.co/{repo_id}/tree/v3")

Copy the output of the next cell and paste it into the chat.

In [ ]:
print(json.dumps(json.load(open("reports/phobert_results_v3.json")), ensure_ascii=False))